# Lab 0 — Environment Check
**Pre-class | ~10 minutes | Colab CPU | No API key**

This notebook is a **smoke test**, not a modeling lab. Run it before Day 1 so a missing package or a broken runtime does not eat lab time tomorrow.

By the end you will have:

1. Installed the libraries this course uses all week
2. Turned a sentence into **tokens** (what models actually read)
3. Turned a sentence into an **embedding** (a vector of meaning)
4. Stored two facts in **ChromaDB** and retrieved the closer one
5. Confirmed the OpenAI and Gradio libraries import — without calling a paid API

> **The key idea:** An LLM app is a stack, not a single library. Today you prove each layer loads. Tomorrow you start using it.

---

## How this course is put together

```
Pre-class   Lab 0     this notebook (runtime + stack check)
Day 1 AM    Labs 1A–3 stack, tools, prompting, inspect a local model
Day 1 PM    Lab 4     quantize + LoRA on a T4 GPU
Day 2 AM    Labs 5–6  serve an OpenAI-compatible API, then RAG
Day 2 PM    Lab 7     Gradio app + capstone
```

Labs 8–12 are optional production-readiness work after class.

## How to run this notebook

1. Open it in Colab from the lab README badge (or **File → Open notebook → GitHub**).
2. Runtime type: **CPU**. You do not need a GPU today.
3. Run cells **in order**. Read the short "why" above each cell before you run it.
4. You are done when the last code cell prints `Environment check PASSED`.


---

## 1. Install the course packages

**Why:** Colab starts empty. Each lab installs what it needs, but Lab 0 installs the set you will see all week so a surprise missing import does not show up in Lab 1.

**What:** `uv` is a fast installer. Colab does not ship it, so we install `uv` first. Colab has no virtual environment, so `uv pip` needs `--system` there.

**When:** Run this once per new Colab runtime. If you already installed from `requirements.txt` locally, you can skip it.


In [ ]:
import shutil
import sys

IN_COLAB = "google.colab" in sys.modules
print(f"Python {sys.version.split()[0]}")
print("Runtime:", "Google Colab" if IN_COLAB else "local Jupyter")

if IN_COLAB:
    # Colab has no virtualenv — uv needs --system here.
    !pip install -q uv
    !uv pip install -q --system transformers torch sentence-transformers chromadb openai langchain langchain-openai gradio
elif shutil.which("uv"):
    !uv pip install -q transformers torch sentence-transformers chromadb openai langchain langchain-openai gradio
else:
    # Local Jupyter without uv — pip is fine.
    !pip install -q transformers torch sentence-transformers chromadb openai langchain langchain-openai gradio

print("Packages installed.")


**Checkpoint:** you should see `Packages installed.` The pip/`uv` log above it can look noisy. That is normal.

If this cell errors, read the message once, then try **Runtime → Restart session** and run from here again.


---

## 2. Import the stack and print versions

**Why:** An install can "succeed" and still leave a broken import. Importing is the real test.

**What:** We load each library and print its version. We do **not** call OpenAI. There is no key in this lab.

**When:** After every fresh install, before you spend time downloading models.


In [ ]:
import sys
import torch
import transformers
import chromadb
import openai
import langchain
import gradio as gr
from openai import OpenAI
from langchain_openai import ChatOpenAI
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

print(f"Python              {sys.version.split()[0]}")
print(f"torch               {torch.__version__}   CUDA available: {torch.cuda.is_available()}")
print(f"transformers        {transformers.__version__}")
print(f"chromadb            {chromadb.__version__}")
print(f"openai              {openai.__version__}")
print(f"langchain           {langchain.__version__}")
print(f"gradio              {gr.__version__}")
print("sentence-transformers imported")
print("ChatOpenAI imported (we will not call it today)")
print()
print("All imports succeeded.")


**Checkpoint:** `CUDA available: False` is expected on Colab CPU. That is correct for Labs 0–3. Lab 4 is the first notebook that needs a T4 GPU.

`ChatOpenAI` and `OpenAI` are imported only to prove the packages exist. Instantiating a client without a key would fail — we wait until Lab 1A for that.


---

## 3. Tokens: what a model actually reads

**Why:** Models do not read English. They read integers. Token count is what you pay for on an API, and what fills a model's context window.

**What:** Load the tiny `gpt2` tokenizer (Lab 1A uses the same model as a teaching demo) and split one sentence.

**When:** Any time you need to know "how long is this prompt, really?"


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")

text = "Deploy the model to production."
ids = tokenizer.encode(text)
tokens = tokenizer.convert_ids_to_tokens(ids)

print("Text:   ", text)
print("Tokens: ", tokens)
print("IDs:    ", ids)
print(f"Count:   {len(ids)} tokens")
print()
print("Notice: whitespace and punctuation are their own pieces.")
print("A 'word' in English is not always one token.")


**Checkpoint:** you get a short list of token strings and matching integer IDs. Lab 3 will show how those IDs enter a real model. Lab 1A peels back HuggingFace `pipeline()` using this same `gpt2` tokenizer.


---

## 4. Embeddings: meaning as a vector

**Why:** Search in RAG is not keyword matching. Each chunk of text becomes a list of numbers. Similar meaning → nearby vectors. This is how Lab 6 finds the right paragraph for a question.

**What:** `all-MiniLM-L6-v2` is a small, local embedding model. No API key. First run downloads about 80 MB.

**When:** Retrieval, semantic cache (Lab 9), and "are these two questions the same?" problems.


In [ ]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

sentence = "Quantization shrinks GPU memory."
vec = embed_model.encode(sentence)

print(f"Sentence:   {sentence}")
print(f"Dimension:  {len(vec)}")          # MiniLM-L6 is 384
print(f"First 8:    {vec[:8].round(4)}")
print()
print("These 384 numbers are the sentence's coordinates in meaning-space.")


**Checkpoint:** dimension should be **384**. Those first eight numbers will not be zero. You now have a local embedding model — the same one Labs 6–9 and 11–12 use.


---

## 5. A tiny vector store

**Why:** Embeddings are only useful if you can **store** them and **ask** "what is closest to this query?" ChromaDB is that store. Lab 6 puts a real knowledge base in it. Today we store two sentences.

**What:** Add two facts. Query with a question about memory. The closer sentence should win.

**When:** Any RAG pipeline — after you embed, before you call the LLM.


In [ ]:
client = chromadb.Client()
collection = client.get_or_create_collection("lab0")

collection.upsert(
    ids=["quant", "ui"],
    documents=[
        "Quantization shrinks model memory so a GPU can hold the weights.",
        "Gradio builds a web chat UI in a few lines of Python.",
    ],
)

query = "How do I reduce VRAM?"
result = collection.query(query_texts=[query], n_results=1)
hit = result["documents"][0][0]

print(f"Query:    {query}")
print(f"Closest:  {hit}")
print()
print("No LLM was called. Retrieval is just nearest-neighbor search.")


**Checkpoint:** the closest document should be the **quantization** sentence, not the Gradio one. That is RAG's retrieval step in miniature. Lab 6 adds chunking, a grounded prompt, and a generator. Lab 7 wraps it in a UI.


---

## 6. The hosted-LLM client (import only)

**Why:** Most of this course talks to models through the **OpenAI Python SDK**. The important deployment idea is not "call OpenAI." It is: the same client can point at OpenAI, Groq, vLLM, Ollama, or the FastAPI proxy you will build in Lab 5 — by changing `base_url`.

```python
from openai import OpenAI
client = OpenAI(api_key=YOUR_KEY, base_url="https://api.openai.com/v1")
```

**What we do today:** prove `openai` and `langchain-openai` import. We do **not** create a client and we do **not** spend tokens.

**When you will use a key:** Lab 1A, first thing on Day 1. The instructor shares `OPENAI_API_KEY`. You store it as a Colab Secret (key icon in the left sidebar) — never inside a cell.


In [ ]:
# Import-only check. Creating OpenAI() without a key raises an error
# in current SDKs, so we stop at the import we already ran above.
print("openai.OpenAI is ready to use in Lab 1A")
print("langchain_openai.ChatOpenAI is ready to use in Lab 1A")
print("gradio is ready to use in Lab 7")
print()
print("No API call was made. No key was required.")


---

## 7. Final report

If every cell above ran, the stack this course needs is present. Accounts and GPU come later, on the days you actually need them.


In [ ]:
print("=" * 56)
print("Environment check PASSED")
print("=" * 56)
print()
print("You are ready for Day 1.")
print()
print("Bring these later — not today:")
print("  Day 1  Labs 1A, 1B, 2, 5, 6, 7")
print("          OPENAI_API_KEY  (instructor provides; Colab Secret)")
print("  Day 1  Lab 4")
print("          Runtime → Change runtime type → T4 GPU")
print("  Day 2  Lab 5")
print("          NGROK_AUTH_TOKEN  (free account at ngrok.com; Colab Secret)")
print()
print("Never paste a key into a notebook cell.")
print()
print("Next: Lab 1A — HuggingFace internals, OpenAI SDK, base_url swap.")


---

## Troubleshooting

| Error | What it usually means | Fix |
|---|---|---|
| `uv: command not found` | `uv` did not install | Re-run the install cell. On Colab it starts with `pip install -q uv`. |
| `No virtual environment found` | `uv pip` on Colab without `--system` | The install cell already passes `--system` on Colab. Re-run it from the top. |
| `ModuleNotFoundError` | Install cell skipped or runtime restarted | Re-run the install cell, then the import cell. |
| Tokenizer or MiniLM download hangs | Hugging Face Hub is slow | Wait 1–2 minutes on first run. Re-run that one cell if it drops. |
| Session disconnected | Colab idle timeout | **Runtime → Reconnect**, then start from the install cell. |
| `CUDA available: True` on Lab 0 | You enabled a GPU early | Fine, but you do not need it until Lab 4. CPU is enough today. |

---

## What you just proved

```
text  →  tokenizer   →  token IDs          (Labs 1A, 3)
text  →  MiniLM      →  384-d vector       (Labs 6–9)
vectors → ChromaDB   →  nearest chunk      (Labs 6, 7)
OpenAI SDK import    →  ready for base_url (Labs 1A, 5)
```

That is the skeleton of every deployment in this course. Lab 1A puts a model and a hosted API on top of it.
